# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading, exploring, and processing the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print summary information about the dataset
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
if hasattr(dataset.metadata, 'keywords'):
    print(f"Keywords: {', '.join(dataset.metadata.keywords)}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and their fields with `@id`s.

A [Croissant RecordSet](https://mlcommons.github.io/croissant/1.0/record-set.html) is a logical group of related records (think: data tables). Each field (column) also has a stable `@id`. All referencing below is by `@id`.

In [ ]:
# Explore the available record sets and their fields using their '@id's

record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for idx, rs in enumerate(record_sets):
    print(f"  {idx+1}. Record Set Name: {rs.name}")
    print(f"     @id: {rs.id}")
    if hasattr(rs, 'description') and rs.description:
        print(f"     Description: {rs.description}")
    # List fields
    if hasattr(rs, 'fields') and rs.fields:
        print("     Fields:")
        for field in rs.fields:
            print(f"        - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this demo, we'll use the main protein abundance record set.

In [ ]:
# List all record set IDs for use
record_set_ids = [rs.id for rs in record_sets]
print(f"Record Set IDs detected: {record_set_ids}")

# We'll use the main record set (the first/only one for this dataset)
main_record_set_id = record_set_ids[0]
print(f"Using record set: {main_record_set_id}\n")

# Load the records as DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

print("Columns in main table:")
print(list(df.columns))

df.head()

## 4. Exploratory Data Analysis (EDA)
We'll filter, normalize, and group some relevant numeric fields. All columns are referenced by their Croissant `@id`s.

**Example fields:**
- `accession` (`cr:accession`)
- `description` (`cr:description`)
- `coverage_percent` (`cr:coverage_percent`)
- `peptide_count` (`cr:peptide_count`)
- `mw` (`cr:mw`)
- ... etc.

Below, we'll examine the molecular weight (`cr:mw`) and group by the description (`cr:description`).

In [ ]:
# Choose Croissant field @ids for analysis
numeric_field_id = 'cr:mw'                # Molecular Weight
group_field_id = 'cr:description'         # Protein Description

# Filter records for non-null and numeric molecular weights
df = df[df[numeric_field_id].notnull()]
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove outliers: Only consider plausible MWs (e.g., > 1000 Da and < 1,000,000 Da)
df_filtered = df[(df[numeric_field_id] > 1000) & (df[numeric_field_id] < 1e6)]
print(f"Filtered records with 1000 < {numeric_field_id} < 1e6:")
print(df_filtered[[numeric_field_id, group_field_id]].head())

# Normalize MW column (z-score normalization)
normalized_col = numeric_field_id + '_normalized'
df_filtered[normalized_col] = (df_filtered[numeric_field_id] - df_filtered[numeric_field_id].mean())/df_filtered[numeric_field_id].std()
print(f"\nFirst few normalized values for {numeric_field_id}:")
print(df_filtered[[numeric_field_id, normalized_col]].head())

# Group by description and show mean MW per group, if available
if group_field_id in df_filtered.columns:
    grouped = df_filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped (mean MW) by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Visualize the distribution of molecular weight (`cr:mw`) and top occurring protein descriptions.

In [ ]:
# Histogram of molecular weights
plt.figure(figsize=(8,4))
plt.hist(df_filtered['cr:mw'].dropna(), bins=50, color='skyblue', edgecolor='k')
plt.xlabel('Molecular Weight (cr:mw)')
plt.ylabel('Frequency')
plt.title('Distribution of Molecular Weights (cr:mw)')
plt.grid(True, alpha=0.3)
plt.show()

# Top protein descriptions
top_desc = df_filtered['cr:description'].value_counts().head(10)
plt.figure(figsize=(8,4))
top_desc.plot(kind='barh', color='orange')
plt.xlabel('Count')
plt.ylabel('Protein Description (cr:description)')
plt.title('Top 10 Protein Descriptions')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Conclusion

- The FAIR<sup>2</sup> dataset was loaded and explored using strict referencing by Croissant `@id`s at the record set and field level.
- Molecular weight (`cr:mw`) shows a typical right-skewed distribution, with most proteins clustered at lower molecular weights.
- The data is well structured for analysis of protein properties, modifications, and comparative proteomics workflows.

Further steps may include advanced analysis, integration with external protein databases, or biomarker discovery pipelines.